1. Train 9 ML regressors on all clean bands with `RandomizedSearchCV`
2. Evaluate on held-out test set (MAPE, R²)
3. Rank models using combined metric: −Norm(MAPE) + Norm(R²)
4. Extract top 5 bands from the best-ranked model
5. Re-run best model with top 5 only and compare
6. Export tuned models for use in SMOF

In [ ]:
import pandas as pd
import numpy as np
import warnings
import joblib
import os
warnings.filterwarnings('ignore')

from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.feature_selection import RFE
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.inspection import permutation_importance
from scipy.stats import gaussian_kde

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

## Load Data

In [ ]:
data = pd.read_csv('Data_subset1.csv')
print(f'Shape: {data.shape}')
print(f'Turbidity range: {data["Turbidity"].min():.1f} – {data["Turbidity"].max():.1f} NTU')
data.head()

In [ ]:
feature_cols = [c for c in data.columns if c.startswith('b')]
X = data[feature_cols]
y = data['Turbidity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f'Features: {X.shape[1]}, Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## Define 9 ML Regressors & Hyperparameter Grids

In [ ]:
models_config = {
    'RandomForest': {
        'model': RandomForestRegressor(random_state=42),
        'params': {
            'n_estimators': [100, 200, 500, 1000],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    'DecisionTree': {
        'model': DecisionTreeRegressor(random_state=42),
        'params': {
            'max_depth': [None, 5, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    'ExtraTrees': {
        'model': ExtraTreesRegressor(random_state=42),
        'params': {
            'n_estimators': [100, 200, 500],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5, 10]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingRegressor(random_state=42),
        'params': {
            'n_estimators': [100, 200, 500],
            'max_depth': [3, 5, 10],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'min_samples_split': [2, 5]
        }
    },
    'LightGBM': {
        'model': LGBMRegressor(random_state=42, verbose=-1),
        'params': {
            'n_estimators': [100, 200, 500],
            'max_depth': [-1, 5, 10, 20],
            'learning_rate': [0.01, 0.05, 0.1],
            'num_leaves': [31, 50, 100]
        }
    },
    'XGBoost': {
        'model': XGBRegressor(random_state=42, verbosity=0),
        'params': {
            'n_estimators': [100, 200, 500],
            'max_depth': [3, 5, 10],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'subsample': [0.8, 1.0]
        }
    },
    'CatBoost': {
        'model': CatBoostRegressor(random_seed=42, verbose=0),
        'params': {
            'iterations': [100, 200, 500],
            'depth': [4, 6, 8, 10],
            'learning_rate': [0.01, 0.05, 0.1]
        }
    },
    'SVR': {
        'model': SVR(kernel='rbf'),
        'params': {
            'C': [0.1, 1, 10, 100],
            'gamma': ['scale', 'auto', 0.01, 0.1],
            'epsilon': [0.01, 0.1, 0.2]
        }
    }
}

print(f'{len(models_config)} models defined (+ RFE = 9 total)')

## Hyperparameter Tuning & Evaluation on Test Set

In [ ]:
results = []
best_models = {}

for name, config in models_config.items():
    print(f'Training: {name} ...', end=' ')
    search = RandomizedSearchCV(
        config['model'], config['params'],
        n_iter=30, cv=10,
        scoring='neg_mean_absolute_percentage_error',
        random_state=42, n_jobs=-1
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_

    y_pred = search.best_estimator_.predict(X_test)
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results.append({'Model': name, 'MAPE (%)': round(mape, 2), 'R²': round(r2, 4),
                    'RMSE': round(rmse, 4), 'Best Params': str(search.best_params_)})
    print(f'MAPE={mape:.2f}%  R²={r2:.4f}')

# RFE
print(f'Training: RFE ...', end=' ')
rfe = RFE(estimator=RandomForestRegressor(n_estimators=200, random_state=42), n_features_to_select=5)
rfe.fit(X_train, y_train)
best_models['RFE'] = rfe

y_pred_rfe = rfe.predict(X_test)
mape_rfe = mean_absolute_percentage_error(y_test, y_pred_rfe) * 100
r2_rfe = r2_score(y_test, y_pred_rfe)
results.append({'Model': 'RFE', 'MAPE (%)': round(mape_rfe, 2), 'R²': round(r2_rfe, 4),
                'RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred_rfe)), 4),
                'Best Params': 'n_features_to_select=5'})
print(f'MAPE={mape_rfe:.2f}%  R²={r2_rfe:.4f}')

## Model Performance Ranking

Combined metric: **−Norm(MAPE) + Norm(R²)** (higher = better)

In [ ]:
results_df = pd.DataFrame(results)
scaler = MinMaxScaler()
norm_mape = scaler.fit_transform(results_df[['MAPE (%)']].values).flatten()
norm_r2 = scaler.fit_transform(results_df[['R²']].values).flatten()
results_df['Performance'] = np.round(-norm_mape + norm_r2, 4)
results_df = results_df.sort_values('Performance', ascending=False).reset_index(drop=True)

print('MODEL PERFORMANCE RANKING (all bands)')
print(results_df[['Model', 'MAPE (%)', 'R²', 'Performance']].to_string(index=False))

best_model_name = results_df.iloc[0]['Model']
print(f'\n★ Best model: {best_model_name}')

## Feature Importance Extraction

Tree-based → `.feature_importances_`, SVR (RBF) → permutation importance

In [ ]:
all_importances = {}

for name, model in best_models.items():
    if name == 'RFE':
        imp = 1.0 / model.ranking_.astype(float)
        imp = imp / imp.sum()
    elif name == 'SVR':
        perm = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)
        imp = np.clip(perm.importances_mean, 0, None)
        if imp.sum() > 0:
            imp = imp / imp.sum()
    else:
        imp = model.feature_importances_
    all_importances[name] = imp

print('Feature importances extracted for all 9 models.')

## Top 5 Bands (from Best Model)

In [ ]:
best_imp = all_importances[best_model_name]
top5_indices = np.argsort(best_imp)[-5:][::-1]
top5_bands = [feature_cols[i] for i in top5_indices]
top5_importances = best_imp[top5_indices]

print(f'Top 5 bands extracted from: {best_model_name}')
print(f'{"Rank":<6}{"Band":<10}{"Importance":<12}')
for rank, (band, imp) in enumerate(zip(top5_bands, top5_importances), 1):
    print(f'{rank:<6}{band:<10}{imp:.6f}')

## Visualisation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

ax = axes[0]
colours_all = ['#e74c3c' if i in top5_indices else '#3498db' for i in range(len(best_imp))]
ax.bar(range(len(best_imp)), best_imp, color=colours_all, width=1.0)
ax.set_xlabel('Band Index')
ax.set_ylabel('Feature Importance')
ax.set_title(f'{best_model_name} Feature Importance — All {len(feature_cols)} Bands')
for idx in top5_indices:
    ax.annotate(feature_cols[idx], xy=(idx, best_imp[idx]),
                fontsize=8, ha='center', va='bottom', color='#c0392b', fontweight='bold')

ax2 = axes[1]
model_names = results_df['Model'].tolist()
x_pos = np.arange(len(model_names))
w = 0.35
ax2.bar(x_pos - w/2, results_df['MAPE (%)'], w, label='MAPE (%)', color='#e74c3c', alpha=0.8)
ax2t = ax2.twinx()
ax2t.bar(x_pos + w/2, results_df['R²'], w, label='R²', color='#27ae60', alpha=0.8)
ax2.set_xlabel('Model')
ax2.set_ylabel('MAPE (%)', color='#e74c3c')
ax2t.set_ylabel('R²', color='#27ae60')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(model_names, rotation=45, ha='right', fontsize=9)
ax2.set_title('Model Comparison')
ax2.legend(loc='upper left'); ax2t.legend(loc='upper right')

plt.tight_layout()
plt.savefig('feature_importance_results.png', dpi=150, bbox_inches='tight')
plt.show()

## Re-Run Best Model with Top 5 Bands

In [ ]:
X_top5_train = X_train[top5_bands]
X_top5_test = X_test[top5_bands]

best_model_all = best_models[best_model_name]

from sklearn.base import clone
best_model_top5 = clone(best_model_all)
best_model_top5.fit(X_top5_train, y_train)

mape_all = mean_absolute_percentage_error(y_test, best_model_all.predict(X_test)) * 100
r2_all = r2_score(y_test, best_model_all.predict(X_test))
mape_t5 = mean_absolute_percentage_error(y_test, best_model_top5.predict(X_top5_test)) * 100
r2_t5 = r2_score(y_test, best_model_top5.predict(X_top5_test))

print(f'Model: {best_model_name}')
print(f'{"":<15}{"All bands":<15}{"Top 5":<15}{"Change"}')
print(f'{"MAPE (%)":<15}{mape_all:<15.2f}{mape_t5:<15.2f}{mape_t5-mape_all:+.2f}')
print(f'{"R²":<15}{r2_all:<15.4f}{r2_t5:<15.4f}{r2_t5-r2_all:+.4f}')

## Actual vs Predicted (Density Scatter)

In [ ]:
y_pred_t5 = best_model_top5.predict(X_top5_test)
xy = np.vstack([y_test.values, y_pred_t5])
density = gaussian_kde(xy)(xy)
order = density.argsort()

fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(y_test.values[order], y_pred_t5[order], c=density[order],
                cmap='viridis', s=30, edgecolors='none')
lims = [min(y_test.min(), y_pred_t5.min()) - 5, max(y_test.max(), y_pred_t5.max()) + 5]
ax.plot(lims, lims, 'k--', alpha=0.4)
ax.set_xlabel('Actual Turbidity (NTU)')
ax.set_ylabel('Predicted Turbidity (NTU)')
ax.set_title(f'{best_model_name} (Top {len(top5_bands)} bands)\nMAPE={mape_t5:.2f}%  R²={r2_t5:.4f}')
ax.set_xlim(lims); ax.set_ylim(lims); ax.set_aspect('equal'); ax.grid(alpha=0.2)
plt.colorbar(sc, ax=ax, label='Density', shrink=0.8)
plt.tight_layout()
plt.savefig('best_model_top5_density.png', dpi=400, bbox_inches='tight')
plt.show()

## Save Results & Export Tuned Models

In [ ]:
results_df.to_csv('model_performance_ranking.csv', index=False)
pd.DataFrame(all_importances, index=feature_cols).to_csv('all_feature_importances.csv')
pd.DataFrame({'Rank': range(1,6), 'Band': top5_bands, 'Importance': top5_importances}).to_csv('top5_bands.csv', index=False)

os.makedirs('tuned_models', exist_ok=True)
for name, model in best_models.items():
    joblib.dump(model, f'tuned_models/{name}.joblib')

print('Saved: model_performance_ranking.csv, all_feature_importances.csv, top5_bands.csv')
print(f'Saved {len(best_models)} tuned models to tuned_models/')
print(f'Best model: {best_model_name}')
print(f'Top 5 bands for SMOF: {top5_bands}')